# EDS Colab Processing Server
## Emotion Data Studio — GPU Processing on Google Colab

**Chay tren Colab Pro (GPU T4/V100/A100)**: Pipeline AI toc do cao, truy cap dashboard tu trinh duyet bat ky dau qua ngrok.

**Sau khi chay xong**: Dashboard se co URL ngrok o cell cuoi cung.

In [2]:
# ============================================================
# 0. CAI DAT MOI TRUONG
# ============================================================

# NOTE: requirements.txt tren GitHub KHONG chua google-cloud-aiplatform.
# Cai dat thu cong cac package bi thieu.

!pip install -q --upgrade pip

# 1. Deep Learning core — phai truoc transformers
!pip install -q torch torchaudio torchvision

# 2. Google AI packages (bi thieu trong requirements.txt)
!pip install -q \
  google-cloud-aiplatform>=2.0.0 \
  google-cloud-storage>=2.14.0 \
  google-genai>=0.8.0 \
  google-auth>=2.0.0 \
  google-auth-httplib2>=0.2.0 \
  cloud-sql-python-connector[pg8000]>=1.0.0

# 3. AI models (bang --no-deps vi co the bi conflict voi torch cu)
!pip install -q --no-deps \
  openai-whisper>=20231117 \
  faster-whisper>=1.0.0 \
  deepface>=0.0.89 \
  facenet-pytorch>=2.5.3 \
  mediapipe>=0.10.8 \
  PySide6>=6.7.0

# 4. scenedetect (tach rieng vi co cu phap [opencv])
!pip install -q "scenedetect[opencv]>=0.6.2"

# 5. Tat ca cac package con lai
!pip install -q \
  transformers>=4.36.0 \
  underthesea>=6.8.0 \
  librosa>=0.10.1 \
  soundfile>=0.12.1 \
  opencv-python>=4.8.0 \
  Pillow>=10.0.0 \
  yt-dlp>=2024.1.0 \
  flask>=3.0.0 \
  flask-cors>=4.0.0 \
  fastapi>=0.110.0 \
  uvicorn>=0.28.0 \
  pydantic-settings>=2.2.0 \
  sqlalchemy>=2.0.0 \
  python-multipart>=0.0.9 \
  psycopg2-binary>=2.9.9 \
  numpy>=1.24.0 \
  pandas>=2.0.0 \
  scikit-learn>=1.3.0 \
  matplotlib>=3.7.0 \
  seaborn>=0.12.0 \
  tqdm>=4.65.0 \
  tensorboard>=2.14.0 \
  pytest>=7.4.0

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — processing will be slow. Use a GPU runtime.")


GPU: Tesla T4
Memory: 15.6 GB


In [3]:
# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
EDS_ROOT = '/content/drive/MyDrive/EDS'
os.makedirs(EDS_ROOT, exist_ok=True)
os.makedirs(f'{EDS_ROOT}/videos', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/clips', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/audio', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/features', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/exports', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/data', exist_ok=True)

print(f"Drive mounted: {EDS_ROOT}")

Mounted at /content/drive
Drive mounted: /content/drive/MyDrive/EDS


In [4]:
# ============================================================
# 2. CLONE / PULL EDS REPO
# ============================================================

REPO_URL = "https://github.com/Kandesfx/Emotion-Data-Studio.git"
REPO_DIR = '/content/Emotion-Data-Studio'

import subprocess, os

if os.path.exists(REPO_DIR):
    print("Repo da ton tai — pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=False)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

import sys
sys.path.insert(0, REPO_DIR)

print(f"Repo ready: {REPO_DIR}")

Cloning repo...
Repo ready: /content/Emotion-Data-Studio


In [8]:
# ============================================================
# 3. CAU HINH GOOGLE CLOUD & GEMINI AUTO-LABELER
# ============================================================
#
# 1. Tao service account tai: https://console.cloud.google.com/iam-admin/serviceaccounts
#    Roles: Storage Admin + Vertex AI User
# 2. Tai JSON key -> upload len Drive: /content/drive/MyDrive/EDS/credentials/
# 3. Tao GCS bucket: gs://your-bucket-name
# 4. Dien GCP_PROJECT_ID, GCS_BUCKET_NAME ben duoi

import os

SERVICE_ACCOUNT_KEY = "/content/drive/MyDrive/EDS/credentials/service-account.json"
if os.path.exists(SERVICE_ACCOUNT_KEY):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SERVICE_ACCOUNT_KEY
    print(f"Service account: OK")
else:
    print("WARNING: Service account key chua co!")

os.environ["GCP_PROJECT_ID"] = "your-gcp-project-id"
os.environ["GCP_LOCATION"] = "us-central1"
os.environ["GCS_BUCKET_NAME"] = "your-bucket-name-emotion-data"

os.environ["GEMINI_MODEL"] = "gemini-2.5-flash"
os.environ["GEMINI_TEMPERATURE"] = "0.2"
os.environ["GEMINI_MAX_TOKENS"] = "8192"
os.environ["GEMINI_INTENSITY_THRESHOLD"] = "0.6"

os.environ["EDS_DATA_DIR"] = f"{EDS_ROOT}/data"
os.environ["EDS_DOWNLOAD_MODE"] = "balanced"
os.environ["EDS_DOWNLOAD_MAX_HEIGHT"] = "720"

try:
    from backend.services.gemini_auto_labeler import GeminiAutoLabeler
    labeler = GeminiAutoLabeler()
    print(f"Gemini: {labeler.status()}")
except Exception as e:
    print(f"WARNING: Gemini config error: {e}")

print("Google Cloud & Gemini configured")

Service account: OK
Gemini: {'configured': False, 'message': "Loi: ('invalid_scope: Invalid OAuth scope or ID token audience provided.', {'error': 'invalid_scope', 'error_description': 'Invalid OAuth scope or ID token audience provided.'})", 'model': 'gemini-2.5-flash', 'location': 'us-central1', 'temperature': 0.2, 'max_output_tokens': 8192, 'agent_runtime': {'enabled': False, 'url': None, 'api_key_set': False}}
Google Cloud & Gemini configured


In [ ]:
# ============================================================
# 4. PREWARM MODELS (GPU)
# ============================================================

import sys, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('EDS')

# Init database
from backend.database.local_db import init_database, get_session
init_database()
print("Database initialized")

# Prewarm core models (skip text/audio emotion for now — loaded lazily)
from backend.ai_models.model_manager import model_manager

logger.info("Prewarming Whisper (medium)...")
try:
    model_manager.prewarm_models(['whisper', 'deepface', 'mtcnn'])
    logger.info("Core models loaded")
except Exception as e:
    logger.warning(f"Some models failed: {e}")

# Load text/audio emotion models (optional)
logger.info("Loading text/audio emotion models...")
try:
    model_manager.prewarm_models(['text_emotion', 'audio_emotion'])
    logger.info("Emotion models loaded")
except Exception as e:
    logger.warning(f"Emotion models failed: {e}")

print(model_manager.status())

In [ ]:
# ============================================================
# 5. KHOI DONG WEB DASHBOARD
# ============================================================
# Entry point: app_cloud:app (FastAPI REST API, serves dashboard)
# web.main:app is for the desktop app (PySide6), not Colab

import subprocess, threading, time

NGROK_TOKEN = "YOUR_NGROK_TOKEN"  # <-- THAY DOI

if NGROK_TOKEN == "YOUR_NGROK_TOKEN":
    print("WARNING: Chua dat NGROK_TOKEN!")
    print("   Lay token tai: https://dashboard.ngrok.com/get-started/your-authtoken")
    public_url = None
else:
    get_ipython().system('pip install -q pyngrok')
    from pyngrok import ngrok
    ngrok.kill()
    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(addr="8765", proto="http", bind_tls=True)
    public_url = tunnel.public_url
    print(f"Dashboard: {public_url}")

import os
os.chdir(REPO_DIR)

def run_server():
    import uvicorn
    uvicorn.run(
        "app_cloud:app",
        host="0.0.0.0",
        port=8765,
        log_level="info",
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("Web server started on port 8765")

In [ ]:
# ============================================================
# 6. START GPU WORKER (BACKGROUND)
# ============================================================
# Worker loop: claim job from local backend → process → report
# Requires local backend running + cloudflared tunnel URL

import threading, time, os

# Paste your cloudflared tunnel URL here (from local PC running cloudflared tunnel)
CLOUDFLARED_URL = "YOUR_CLOUDFLARED_URL"  # <-- THAY DOI: https://abc123.trycloudflare.com

WORKER_DATA_DIR = f"{EDS_ROOT}/data"
WORKER_VIDEO_DIR = f"{EDS_ROOT}/videos"
WORKER_CLIP_DIR = f"{EDS_ROOT}/clips"
WORKER_AUDIO_DIR = f"{EDS_ROOT}/audio"

for d in [WORKER_DATA_DIR, WORKER_VIDEO_DIR, WORKER_CLIP_DIR, WORKER_AUDIO_DIR]:
    os.makedirs(d, exist_ok=True)
    os.environ.setdefault(f"EDS_{d.split('/')[-1].upper()}_DIR", d)

os.environ["EDS_DATA_DIR"] = WORKER_DATA_DIR

def run_worker():
    import sys
    sys.path.insert(0, REPO_DIR)
    import uuid, socket
    import logging
    logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
    log = logging.getLogger("ColabWorker")

    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

    from colab.colab_worker import run_worker_loop
    worker_id = f"colab-{socket.gethostname()[:8]}-{uuid.uuid4().hex[:6]}"

    log.info(f"GPU: {gpu_name} ({gpu_mem:.1f}GB)")

    if CLOUDFLARED_URL == "YOUR_CLOUDFLARED_URL":
        log.error("Chua dat CLOUDFLARED_URL!")
        log.error("1. Tren PC: chay 'cloudflared tunnel --url http://localhost:8765'")
        log.error("2. Copy URL (https://xxx.trycloudflare.com) vao CLOUDFLARED_URL o tren")
        return

    run_worker_loop(
        backend_base_url=CLOUDFLARED_URL,
        worker_id=worker_id,
        gpu_name=gpu_name,
        gpu_memory_gb=gpu_mem,
        repo_dir=REPO_DIR,
        data_dir=WORKER_DATA_DIR,
    )

worker = threading.Thread(target=run_worker, daemon=True)
worker.start()
print("GPU worker started")

In [ ]:
# ============================================================
# 7. HUONG DAN SU DUNG
# ============================================================

print("=" * 60)
print("EMOTION DATA STUDIO — COLAB READY")
print("=" * 60)
print()
if 'public_url' in dir() and public_url:
    print(f"DASHBOARD: {public_url}")
    print()
print("Cac buoc tiep theo:")
print("1. Mo dashboard tren trinh duyet")
print("2. Nhap video URL de thu hoach")
print("3. Theo doi tien trinh xu ly")
print("4. Review va xuat dataset")
print()
print(f"Data dir: {EDS_ROOT}")
print(f"Videos:   {EDS_ROOT}/videos")
print(f"Clips:    {EDS_ROOT}/clips")
print(f"Exports:  {EDS_ROOT}/exports")